# Meqpy Tutorial — 3. Simulating Experiments

[← Previous: 2. System and States](02_System_and_States.ipynb) | [🏠 Index](00_Overview.ipynb) | [Next: 4. BandSystem and BandTransitions →](04_BandSystem_BandTransitions.ipynb)

In [ ]:
import meqpy

import numpy as np
import matplotlib.pyplot as plt

### Overview

- [3. Simulating Experiments](#sim_exp)
    - [3.1. I(V): Bias Spectroscopy](#sim_exp_iv)
        - [3.1.1 Define System](#sim_exp_iv_def_system)
        - [3.1.2 Build Rate Matrix](#sim_exp_iv_build_matrix)
        - [3.1.3 Solve Master Equation](#sim_exp_iv_solve_matrix)
        - [3.1.4 Defining Measurement Operator](#sim_exp_iv_def_measurement)
        - [3.1.5 Perform Measurement](#sim_exp_iv_do_measurement)
    - [3.2 I(z): z-Spectroscopy](#sim_exp_iz)
    - [3.3 STML](#sim_exp_stml)
    - [3.4 Photo Current](#sim_exp_photocurrent)
    - [3.5 Custom Lineshape](#sim_exp_customlineshape)
        - [3.5.1 Pylaron](#sim_exp_customlineshape_pylaron)
        - [3.5.2 Various Lineshapes](#sim_exp_customlineshape_various)

<a id='sim_exp'></a>
## 3. Simulating Experiments

Using the ``System`` class, one can simulate various experiments by following these steps:
- define ``System`` with all necessary ``States``
- build rate matrix ``W``, including all transition rates
- solve the master equation for equilibrium to obtain occupation vector ``P``
- define measurement operator ``C`` for whichever observable the experiment is measuring
- apply measurement operator ``C`` to occupation vector ``P`` to perform the measurement

The measurement operator depends on the observable to be measured and is most often derived from one component of the hopping matrix.

In the following we will perform various measurement techniques, typically used in STS, for small example systems.

<a id='sim_exp_iv'></a>
### 3.1 I(V) Spectroscopy

Let us now perform I(V) spectroscopy, where the tunneling current is measured in dependence of the applied bias voltage.

<a id='sim_exp_iv_def_system'></a>
#### 3.1.1 Define System

For this, we use a simple three state system with:
- a neutral ground state ("S0")
- a positively charged doublet state at energy 1.0eV ("D0+")
- a negatively charged doublet state with energy 0.7eV ("D0-").

In addition, we will use a gaussian broadening of 100meV for the lineshape and assume a reorganization energy of 250meV.

As described in chapter [1.2.2 Choosing Anchor](01_Basics.ipynb) it is always best to define the ground state as first state, since it usually has a non-zero occupation probability.

In [ ]:
system = meqpy.System(hwhm=0.1, reorg_shift=0.25, kappa_mode="constant")

system.states = [
    meqpy.State(label="S0", energy=0.0, charge=0, multiplicity=1),
    meqpy.State(label="D0+", energy=1.0, charge=+1, multiplicity=2),
    meqpy.State(label="D0-", energy=0.7, charge=-1, multiplicity=2),
]

<a id='sim_exp_iv_build_matrix'></a>
#### 3.1.2 Build rate matrix

The system is coupled to two electron baths: The sample, which is strongly coupled to the system, and the tip, which is weakly coupled. In addition, a bias voltage is applied between sample and tip, where we assume for now that the system is at the same bias potential as the sample.

Accordingly, the rate matrix is composed of two submatrices:
1. charge transfer between tip and system, which depends on the bias voltage and tip height
1. charge transfer between sample and system, which does not depend on bias voltage

In the following, we will be referring to these two rate matrices as ``Wt`` and ``Ws`` for charge transitions via tip and via sample, respectively.

In [ ]:
tip_height = 6.0  # Å
sample_distance = 1.0  # Å

bias = np.linspace(-2, 2, 201)  # V

# charge transitions via tip
Wt = system.charging_rates(tip_height, bias)

# charge transitions via sample
Ws = system.charging_rates(sample_distance)

# combine to one rate matrix
W = Wt + Ws
W.shape

<a id='sim_exp_iv_solve_matrix'></a>

#### 3.1.3 Solve Master Equation for Equilibrium

The master equation has to be solved in the equilibrium condition:
```
0 = W @ P
```
with ``P`` being the occupation probability vector.
This can be done using the ``meqpy.solve_equilibrium(W)`` method as described in [1. Basics](01_Basics.ipynb). The diagonal elements of ``W`` will be filled automatically before solving it, thus we do not need to worry about them.

In [ ]:
# solve rate equation to obtain probability vector P
iv_P = meqpy.solve_equilibrium(W)

iv_P.shape

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))

for i, P_state in enumerate(iv_P.T):
    state = system.states[i].label
    ax.plot(bias, P_state, label=state)
ax.set_yscale("log")
ax.set_xlabel("Bias Voltage (V)")
ax.set_ylabel("Occupation Probability")
ax.set_ylim(1e-8, 2.0)
ax.legend(frameon=False)

plt.show()

<a id='sim_exp_iv_def_measurement'></a>
#### 3.1.4 Defining Measurement Operator

The goal of an experiment is to obtain a measurement observable, which depends on the occupation probability vector ``P``. For this, one needs to define a measurement operator ``C``. Here we will be looking at the total tunneling current, thus we need to calculate the total charge flowing from or to the system via one lead. In the following we will be considering the flow of charges between system and sample.

The absolute flow of charges between is simply given by ``Ws`` as defined above. However, the total net current is given by the flow of charges from sample to system minus the flow of charges from system to sample. Thus, one needs to multiply ``Ws`` with a charge difference matrix ``dQ``. To convert the units from 1/s to Ampere, we also multiply with the elementary charge of the electron:

In [ ]:
current_operator = Ws * system.dQ * meqpy.constants.ELEMENTARY_CHARGE

<a id='sim_exp_iv_do_measurement'></a>
#### 3.1.5 Perform Measurement

The observable can now easily be obtained by applying the measurement operator to the occupation probability vector and summation over all elements, using ``meqpy.measurement()``

In [ ]:
iv_current = meqpy.measurement(current_operator, iv_P)

# differentiate to obtain dI/dV signal
iv_didv = np.gradient(iv_current, bias, axis=-1)

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(4, 3)

# dI/dV axis (left)
ax.plot(bias, iv_didv * 1e12, "-", color="k")
ax.set_ylabel("dI/dV (pS)")

# Current axis (right)
ax2 = ax.twinx()
ax2.plot(bias, iv_current * 1e12, "-", color="r")
ax2.yaxis.label.set_color("red")
ax2.tick_params(axis="y", colors="red")
ax2.set_ylabel("Current (pA)")

# x-axis
ax.set_xlabel("Bias Voltage (V)")
plt.show()

<a id='sim_exp_iz'></a>
### 3.2 I(z) Spectroscopy

In the second example we perform I(z) spectroscopy, where the tunneling current is measured in dependence of the tip height, while the bias voltage is kept constant. Let us assume the system is more weakly coupled to the substrate, e.g. by a decoupling layer. In such a case we can reach saturation current at small tip heights, when the coupling to the tip becomes stronger than the coupling to the substrate.

The steps to run the experiments are the same as above, except that this time ``tip_height`` is varied instead of ``bias``:

In [ ]:
# define system with states

system = meqpy.System(hwhm=0.1, reorg_shift=0.25, kappa_mode="constant")

system.states = [
    meqpy.State(label="S0", energy=0.0, charge=0, multiplicity=1),
    meqpy.State(label="D0+", energy=1.0, charge=+1, multiplicity=2),
    meqpy.State(label="D0-", energy=0.7, charge=-1, multiplicity=2),
]

In [ ]:
# Build Rate Matrix

bias = 1.5  # V: held constant
sample_distance = 4.0  # Å: larger sample_distance --> weaker coupling to system

tip_height = np.arange(2.5, 6, 0.2)  # Å

# charge transitions via tip
Wt = system.charging_rates(tip_height, bias)

# charge transitions via sample
Ws = system.charging_rates(sample_distance)

# combine to one rate matrix
W = Wt + Ws
W.shape

In [ ]:
# solve rate equation to obtain probability vector P
iz_P = meqpy.solve_equilibrium(W)

iz_P.shape

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))

for i, P_state in enumerate(iz_P.T):
    state = system.states[i].label
    ax.plot(tip_height, P_state, label=state)
ax.set_yscale("log")
ax.set_xlabel("Tip Height (Å)")
ax.set_ylabel("Occupation Probability")
ax.set_ylim(1e-3, 2.0)
ax.legend(frameon=False)

plt.show()

In [ ]:
# define current operator
current_operator = Ws * system.dQ * meqpy.constants.ELEMENTARY_CHARGE

In [ ]:
# perform measurement
iz_current = meqpy.measurement(current_operator, iz_P)

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(4, 3)

ax.plot(tip_height, iz_current)
ax.set_ylabel("Current (A)")
ax.set_xlabel("Tip Height (Å)")
ax.set_yscale("log")
plt.show()

<a id='sim_exp_stml'></a>
### 3.3 STML

So far we have only considered charging rates, however, any transition between states is possible. Here we will now expand the system to perform STML experiments. For this we use six states: in addition to the old three, we are adding two neutral excited states ("T1" and "S1") as well as one charged excited state ("D1+").

After creating the system, the non-charging transitions have to be defined using the ``System.matrix_by_states()`` method.

The measurement is in this experiment not only limited to current, but we will also be measuring the emission of two radiative decays, thus two more measurement operators need to be defined and applied.

In [ ]:
# define system with states

system = meqpy.System()

system.states = [
    meqpy.State(label="S0", energy=0.0, charge=0, multiplicity=1),
    meqpy.State(label="T1", energy=1.5, charge=0, multiplicity=3),
    meqpy.State(label="S1", energy=1.9, charge=0, multiplicity=1),
    meqpy.State(label="D0+", energy=2.3, charge=+1, multiplicity=2),
    meqpy.State(label="D1+", energy=3.7, charge=+1, multiplicity=2),
    meqpy.State(label="D0-", energy=1.0, charge=-1, multiplicity=2),
]

In [ ]:
# set non-charging transittion rates

# radiative coupling of 1meV --> rate is given by coupling / planck
v_rad = 1e-3 / meqpy.constants.PLANCK_EV

# non-radiative decay, assuming triplet lifetime of 100ns
v_nonrad = 1 / 100e-9

In [ ]:
# build rate matrix

bias = -3.0  # V
sample_distance = 6.0  # Å
tip_height = np.arange(4, 8, 0.2)  # Å

# charge transitions via tip
Wt = system.charging_rates(tip_height, bias)

# charge transitions via sample
Ws = system.charging_rates(sample_distance)

# non-charging transitions
W0 = system.zeros
W0 += system.matrix_by_states("S1", "S0") * v_rad  # radiative decay
W0 += system.matrix_by_states("D1+", "D0+") * v_rad  # radiative decay
W0 += system.matrix_by_states("T1", "S0") * v_nonrad  # non-radiative decay

# combine to one rate matrix
W = Wt + Ws + W0
W.shape

In [ ]:
# solve rate equation to obtain probability vector P
stml_P = meqpy.solve_equilibrium(W)

stml_P.shape

In [ ]:
# define measurement operators and perform measurement

# current operator
current_operator = Ws * system.dQ * meqpy.constants.ELEMENTARY_CHARGE
stml_current = meqpy.measurement(current_operator, stml_P)

# measurement operator for S1 -> S0 emission
s1_operator = system.matrix_by_states("S1", "S0") * v_rad
s1_emission = meqpy.measurement(s1_operator, stml_P)

# measurement operator for D1+ -> D0+ emission
d1_operator = system.matrix_by_states("D1+", "D0+") * v_rad
d1_emission = meqpy.measurement(d1_operator, stml_P)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 3))

ax[0].plot(tip_height, np.abs(stml_current))
ax[0].set_yscale("log")
ax[0].set_ylabel("Current (A)")
ax[0].set_xlabel("Tip Height (Å)")

ax[1].plot(tip_height, s1_emission, label="S1 → S0")
ax[1].plot(tip_height, d1_emission, label="D1+ → D0+")
ax[1].set_yscale("log")
ax[1].set_ylabel("Emission (1/s)")
ax[1].set_xlabel("Tip Height (Å)")
ax[1].legend(frameon=False)

ax[2].plot(np.abs(stml_current), s1_emission, label="S1 → S0")
ax[2].plot(np.abs(stml_current), d1_emission, label="D1+ → D0+")
ax[2].set_yscale("log")
ax[2].set_xscale("log")
ax[2].set_ylabel("Emission (1/s)")
ax[2].set_xlabel("Current (A)")
ax[2].legend(frameon=False)

plt.tight_layout(pad=1)
plt.show()

<a id='sim_exp_photocurrent'></a>
### 3.4 Photocurrent

This example shows in a simple way, how photocurrent experiments may be performed in silico. We use a simple system with only four states and measure the photocurrent in I(V) spectroscopy, for various incident light intensities. To do this, we exploit the broadcasting functionalities of ``np.ndarray``.

In addition we will be assuming a voltage drop over a decoupling layer, such that the system-sample transitions are also bias voltage dependent.

In [ ]:
# define system with states

system = meqpy.System(
    hwhm=0.05,  # eV
    reorg_shift=0.2,  # eV
    kappa_mode="constant",
)

system.states = [
    meqpy.State(label="S0", energy=0.0, charge=0, multiplicity=1),
    meqpy.State(label="S1", energy=1.9, charge=0, multiplicity=1),
    meqpy.State(label="D0+", energy=2.5, charge=+1, multiplicity=2),
    meqpy.State(label="D0-", energy=1.7, charge=-1, multiplicity=2),
]

In [ ]:
# build rate matrix (without excitation)

voltage_drop = 0.1  # which part of the bias voltage drop between system and sample

tip_height = 6.0  # Å
sample_distance = 3.0  # Å
bias = np.linspace(-3.5, 2.5, 251)  # V

# charge transitions via tip including voltage drop
Wt = system.charging_rates(tip_height, bias * (1 - voltage_drop))

# charge transitions via sample including voltage drop
Ws = system.charging_rates(sample_distance, bias * (0 - voltage_drop))

# non-charging relaxation
v_rad = 1e-3 / meqpy.constants.PLANCK_EV  # radiative decay
W0 = system.matrix_by_states("S1", "S0") * v_rad  # radiative decay

# combine to one rate matrix
W = Wt + Ws + W0
W.shape

In [ ]:
# add excitation rate, for various excitation strengths
excitations = np.linspace(0, 1e-3, 6)  # eV

# W_exc is reshaped, to allow for broadcasting functionalies
# when adding to W later
W_exc = system.matrix_by_states("S0", "S1") * excitations.reshape(-1, 1, 1, 1)

W_exc /= meqpy.constants.PLANCK_EV  # eV --> 1/s

W_exc.shape

In [ ]:
# solve rate equation to obtain probability vector P
pc_P = meqpy.solve_equilibrium(W + W_exc)

pc_P.shape

In [ ]:
# define current operator and perform measurement
current_operator = Ws * system.dQ * meqpy.constants.ELEMENTARY_CHARGE
pc_current = meqpy.measurement(current_operator, pc_P)

# differentiate to obtain dI/dV signal
pc_didv = np.gradient(pc_current, bias, axis=-1)

pc_didv.shape  # (num_excitations, num_bias)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))

for i, spec in enumerate(pc_didv):
    label = f"{excitations[i] * 1e3:.1f} meV"
    ax.plot(bias, spec * 1e12, "-", label=label)
ax.set_ylabel("dI/dV (pS)")
ax.set_xlabel("Bias Voltage (V)")
ax.legend(frameon=False, title="Excitation Coupling:")
plt.show()

<a id='sim_exp_customlineshape'></a>
### 3.5 Custom Lineshapes

In the following, we will demonstrate how to use custom lineshapes for the normalized charging transitions. This will be done in two examples: The first will be using a custom lineshape created by the [``pylaron``](https://github.com/NilsKrane/pylaron) package and the second example will show one method to assign different lineshapes to different transitions.

<a id='sim_exp_customlineshape_pylaron'></a>
### 3.5.1 Pylaron

An ionic decoupling layer between molecule and substrate, e.g. few-layer NaCl, will cause a shift of the ion resonance to higher energies, in accordance with the reorganization energy. This effect can be mimicked by shifting a gaussian lineshape using the ``reorg_shift`` parameter. A better approximation for this effect is the use of a polaron model[[1], [2]], as implemented by the [``pylaron``](https://github.com/NilsKrane/pylaron) package.

We assume the spectral function $J(\Omega)$ of the phonons to be a simple rectangular function between $\Omega_\mathrm{min}=18\,\mathrm{meV}$ and $\Omega_\mathrm{max}=31\,\mathrm{meV}$ [[1]], which integrates to a total reorganization energy ``reorg_energy``. The corresponding calculated spectral function $S(E)$ is then being integrated to obtain the normalized charging lineshape. Finally, we create a callable function via the ``interp1d`` function.

[1]: https://doi.org/10.1021/acsnano.4c07136
[2]: https://doi.org/10.1021/acsnano.6c01502

In [ ]:
# create lineshape generator function

import pylaron  # pip install git+https://github.com/NilsKrane/pylaron
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d


def get_polaron_lineshape(
    reorg_energy: float,
    omega_min: float = 18e-3,
    omega_max: float = 31e-3,
    xrange: float = 1.0,
    dx: float = 1e-4,
    lor: float = 1e-4,
    gauss: float = 1e-3,
):
    """Create callable lineshape function, using polaron model and rectangular phonon spectral function J."""

    # create rectangular phonon spectral function J
    Jx = np.arange(0, omega_max + dx, dx)
    Jy = pylaron.Jrect(Jx, reorg_energy, omega_min, omega_max)

    # calculate polaron spectrum
    Sx, Sy = pylaron.spectrum(Jy, dx, xrange, lor=lor, gauss=gauss)
    Sy_integrated = cumulative_trapezoid(Sy, Sx, initial=0)
    Sy_integrated /= np.max(Sy_integrated)

    # make callable interpolation function
    lineshape = interp1d(
        Sx, Sy_integrated, kind="cubic", bounds_error=False, fill_value=(0.0, 1.0)
    )

    return lineshape

In [ ]:
# create custom lineshape callable for reorganization energy of 250meV
polaron_lineshape = get_polaron_lineshape(reorg_energy=250e-3, dx=1e-3)

In [ ]:
# plot lineshape and derivative

x = np.linspace(-0.5, 1.0, 201)
y = polaron_lineshape(x)
dy = np.gradient(y, x)

fig, ax = plt.subplots()
fig.set_size_inches(5, 3)

# charging transition axis axis (left)
ax.plot(x, y, "-", color="k")
ax.set_ylabel("Norm. Charging Transition")

# derivative axis (right)
ax2 = ax.twinx()
ax2.plot(x, dy, "-", color="r")
ax2.yaxis.label.set_color("red")
ax2.tick_params(axis="y", colors="red")
ax2.set_ylabel("Derivative")

# x-axis
ax.set_xlabel("Energy (eV)")
plt.show()

Now we can use our custom lineshape in an experiment. For example a simple I(V) spectroscopy, as described in [chapter 3.1](#sim_exp_iv):

In [ ]:
# define system with states

system = meqpy.System(kappa_mode="constant")

system.states = [
    meqpy.State(label="S0", energy=0.0, charge=0, multiplicity=1),
    meqpy.State(label="D0+", energy=1.0, charge=+1, multiplicity=2),
    meqpy.State(label="D0-", energy=0.7, charge=-1, multiplicity=2),
]

In [ ]:
# use custom lineshape in system
system.lineshape = polaron_lineshape

In [ ]:
# build and solve rate matrix

tip_height = 6.0  # Å
sample_distance = 1.0  # Å

bias = np.linspace(-2, 2, 201)  # V

# charge transitions via tip
Wt = system.charging_rates(tip_height, bias)

# charge transitions via sample
Ws = system.charging_rates(sample_distance)

# combine to one rate matrix
W = Wt + Ws

# solve rate equation to obtain probability vector P
polaron_P = meqpy.solve_equilibrium(W)

In [ ]:
# define current operator and perform measurment

current_operator = Ws * system.dQ * meqpy.constants.ELEMENTARY_CHARGE
polaron_current = meqpy.measurement(current_operator, polaron_P)

# differentiate to obtain dI/dV signal
polaron_didv = np.gradient(polaron_current, bias, axis=-1)

In [ ]:
# display results

fig, ax = plt.subplots()
fig.set_size_inches(6, 3)

# dI/dV axis (left)
ax.plot(bias, polaron_didv * 1e12, "-", color="k")
ax.set_ylabel("dI/dV (pS)")

# Current axis (right)
ax2 = ax.twinx()
ax2.plot(bias, polaron_current * 1e12, "-", color="r")
ax2.yaxis.label.set_color("red")
ax2.tick_params(axis="y", colors="red")
ax2.set_ylabel("Current (pA)")

# x-axis
ax.set_xlabel("Bias Voltage (V)")
plt.show()

<a id='sim_exp_customlineshape_various'></a>
### 3.5.2 Various Lineshapes

In this example we will show one way to implement individual lineshapes for certain transitions, i.e. the lineshape for charging transitions between a pair of states is different from the lineshape between another pair of states. Custom lineshapes are only required to be a callable object that accepts an array of shape ``(M, N, N)`` (``N`` being number of states in the system) and returns an array with the same shape.

**Note**: This very loose handling of custom lineshapes by ``meqpy`` allows for great flexibility, but it is also the responsibility of the user to sanity check the custom function themselves.

In the following example, we will consider a five-state system with one neutral ground state and two states each for the anion and cation state. For the two transitions ("S0", "D0+") and ("S0", "D0-") we will be assigning individual lineshapes, whereas all other charging transitions will use a default lineshape.

For simplicity, the lineshapes will all be of type ``"gaussian"``, as implemented in ``meqpy``, however with different ``hwhm`` parameters.

In [ ]:
# define system with states

system = meqpy.System(kappa_mode="constant", hwhm=0.1, reorg_shift=0.2)

system.states = [
    meqpy.State(label="S0", energy=0.0, charge=0, multiplicity=1),
    meqpy.State(label="D0+", energy=1.0, charge=+1, multiplicity=2),
    meqpy.State(label="D1+", energy=1.5, charge=+1, multiplicity=2),
    meqpy.State(label="D0-", energy=0.7, charge=-1, multiplicity=2),
    meqpy.State(label="D1-", energy=1.2, charge=-1, multiplicity=2),
]

In [ ]:
from meqpy.utils import lineshape_integral


# define special lineshapes
def lineshape_NIR(x):
    return lineshape_integral("gaussian", x=x, hwhm=0.05)


def lineshape_PIR(x):
    return lineshape_integral("gaussian", x=x, hwhm=0.2)


# define default lineshape, based on system.hwhm and system.reorg_shift
def lineshape_default(x):
    return lineshape_integral("gaussian", x=x - system.reorg_shift, hwhm=system.hwhm)


# get indices of states
gs = system.get_index("S0")
d0p = system.get_index("D0+")
d0n = system.get_index("D0-")

# create dictionary for all custom transitions
lineshape_dict = {
    (gs, d0p): lineshape_PIR,
    (gs, d0n): lineshape_NIR,
}


# built callable lineshape function
def my_lineshape(x: np.ndarray) -> np.ndarray:
    # create output array using default lineshape
    out = lineshape_default(x)

    # overwrite values of transitions with custom lineshapes
    for (i, j), lineshape in lineshape_dict.items():
        out[..., i, j] = lineshape(x[..., i, j])
        out[..., j, i] = lineshape(x[..., j, i])

    return out

In [ ]:
# register custom lineshape in system
system.lineshape = my_lineshape

In [ ]:
# build and solve rate matrix as normal

tip_height = 6.0  # Å
sample_distance = 1.0  # Å

bias = np.linspace(-2, 2, 201)  # V

# charge transitions via tip
Wt = system.charging_rates(tip_height, bias)

# charge transitions via sample
Ws = system.charging_rates(sample_distance)

# combine to one rate matrix
W = Wt + Ws

# solve rate equation to obtain probability vector P
custom_P = meqpy.solve_equilibrium(W)

In [ ]:
# define current operator and perform measurment

current_operator = Ws * system.dQ * meqpy.constants.ELEMENTARY_CHARGE
custom_current = meqpy.measurement(current_operator, custom_P)

# differentiate to obtain dI/dV signal
custom_didv = np.gradient(custom_current, bias, axis=-1)

In [ ]:
# display results

fig, ax = plt.subplots()
fig.set_size_inches(6, 3)

# dI/dV axis (left)
ax.plot(bias, custom_didv * 1e12, "-", color="k")
ax.set_ylabel("dI/dV (pS)")

# Current axis (right)
ax2 = ax.twinx()
ax2.plot(bias, custom_current * 1e12, "-", color="r")
ax2.yaxis.label.set_color("red")
ax2.tick_params(axis="y", colors="red")
ax2.set_ylabel("Current (pA)")

# x-axis
ax.set_xlabel("Bias Voltage (V)")
plt.show()

---

[← Previous: 2. System and States](02_System_and_States.ipynb) | [🏠 Index](00_Overview.ipynb) | [Next: 4. BandSystem and BandTransitions →](04_BandSystem_BandTransitions.ipynb)